# 第 3 课 · 请求头与浏览器伪装

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：认识“请求头”，重点学会 **User-Agent（你是谁）**，并看懂为什么自己的 `douban.py` 要伪装浏览器。
> 只引入一个新概念：**请求头 headers**。

In [1]:
# 每个 notebook 开头都要重新 import 这一课要用的库
import requests                                   # HTTP 请求库（每本 notebook 都要重新 import）
print('requests 版本：', requests.__version__)

requests 版本： 2.34.2


## 3.1 请求头是什么：请求附带的“说明便签”

**类比**：你寄一个包裹，除了里面的东西（请求体/网址），快递单上还会印一堆说明：寄件人、包裹类型、是否易碎。

**是什么**：每次 HTTP 请求，除了网址，客户端还会顺带发一个**键值对小字典**，叫 **请求头（Request Headers）**，用来告诉服务器“我是谁、我能读什么格式”之类的附加信息。

你在第 0 课按 F12 看到的 **Request Headers**，就是它。

## 3.2 最重要的一个请求头：`User-Agent`

`User-Agent`（简称 UA）这一行写的是：**“我这个客户端是什么软件”**。

- 你用 Chrome 浏览器访问网站，UA 长这样：`Mozilla/5.0 (Windows NT 10.0; ...) Chrome/120.0.0.0 ...`
- 而 requests 库默认的 UA 是：**`python-requests/2.34.2`**——服务器一眼就能认出“这不是真人浏览器，是个程序”。

先离线看看 requests 默认的 UA 到底长什么样：

In [2]:
from requests.utils import default_headers      # 取 requests 默认会带的请求头
headers = default_headers()
print('requests 默认 User-Agent：', headers['User-Agent'])
# 你会看到类似：python-requests/2.34.2
# 服务器看到这一行，立刻知道“来的是个 Python 脚本”。

requests 默认 User-Agent： python-requests/2.34.2


## 3.3 怎么自己传请求头：`headers=` 参数

`requests.get(url, headers={...})` 接收一个字典，里面的键值对会被附加到请求头上。

`httpbin.org/headers` 会把“你发来的请求头”原样回显，正好验证。联网用 `try/except` 包好：

In [3]:
my_headers = {
    'User-Agent': 'MyDemoCrawler/0.1 (learning)',   # 自己写一个 UA
}

try:
    resp = requests.get(
        'https://httpbin.org/headers',
        headers=my_headers,
        timeout=10
    )
    data = resp.json()
    print('服务器收到的 User-Agent：', data['headers'].get('User-Agent'))
except Exception as e:
    print('这次没联网，不影响学习：', type(e).__name__, e)

服务器收到的 User-Agent： MyDemoCrawler/0.1 (learning)


## 3.4 为什么要伪装成浏览器：对照你自己的 `douban.py`

有些网站不希望被程序抓取，它们的判断逻辑很简单：**看 UA 里有没有 `python-requests`**，有就直接拒绝（返回 403 或 418 之类的非 200 状态码）。

打开你自己的 `爬豆瓣小尝试\douban.py` 看看：它现在**还没有带 headers**，所以很可能被豆瓣直接返回 418。给它加上 headers 之后，就变成下面这样：

```python
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ... Chrome/120.0.0.0 ..."
}
response = requests.get("https://movie.douban.com/top250", headers=headers)
```

下面亲手对比“默认 UA”和“浏览器 UA”请求同一个豆瓣页面的状态码。**这一步联网不一定成功，用 try/except 包好，不强求结果**：

In [4]:
url = 'https://movie.douban.com/top250'
browser_ua = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'     # 抄一份常见 Chrome 的 UA

# 第一次：用 requests 默认 UA
try:
    r1 = requests.get(url, timeout=10)
    print('默认 UA 请求豆瓣，状态码：', r1.status_code)
except Exception as e:
    print('默认 UA 请求失败（没联网或被拦）：', type(e).__name__, e)

# 第二次：伪装成 Chrome 浏览器
try:
    r2 = requests.get(url, headers={'User-Agent': browser_ua}, timeout=10)
    print('伪装 Chrome 后请求豆瓣，状态码：', r2.status_code)
except Exception as e:
    print('伪装 UA 后请求失败（没联网）：', type(e).__name__, e)

默认 UA 请求豆瓣，状态码： 418


伪装 Chrome 后请求豆瓣，状态码： 200


> 合规提醒：这是你之前自己写的学习脚本。**只用于学习，不要高频、重复请求任何网站**，
> 更不要绕过网站明确的 robots.txt 限制——第 6 课会专门讲合规红线。

## 3.5 小结（记住这三句）

1. **请求头**是请求附带的一张“说明便签”，用字典传给 `headers=`。
2. **`User-Agent`** 告诉服务器你是什么软件；requests 默认是 `python-requests/x.x.x`，很容易被认出。
3. 把 UA 改成常见浏览器的字符串，叫**伪装浏览器**；这是绕过“按 UA 拦爬虫”最基础的一招。

## 3.6 随堂小练习（自己动手，空格里写）

1. 用 `requests.get('https://httpbin.org/headers', headers={...})` 发请求，自己加一个叫 `X-My-Name` 的请求头，值随便写，打印回显里它是否出现。
2. 把默认 UA 请求 `https://httpbin.org/headers` 的回显打印出来，对比 3.3 节你自己传的 UA。

In [5]:
# TODO：自己写代码完成上面两题（这是随堂小练，没有单独答案文件）


## 来源与延伸

- requests 官方文档（自定义请求头）：https://requests.readthedocs.io/en/latest/user/quickstart/#custom-headers
- MDN 请求头文档：https://developer.mozilla.org/zh-CN/docs/Web/HTTP/Headers
- User-Agent 字符串说明（MDN）：https://developer.mozilla.org/zh-CN/docs/Web/HTTP/Headers/User-Agent
- httpbin：https://httpbin.org
- 下一课：服务器怎么记住“你来过”——Cookie 与 Session。